In [17]:
import pandas as pd
import random
from collections import defaultdict

random.seed(42)  # 任意のシード

# 定義
participants = [f'P{i+1}' for i in range(24)]
conditions = ['spatial', 'pointing', 'pointing+spatial', 'label']
task_sets = ['A', 'B', 'C', 'D']

# Step 1: 全ての組み合わせを6回ずつ作る（96要素）
all_pairs = [(c, t) for c in conditions for t in task_sets for _ in range(6)]
random.shuffle(all_pairs)

# Step 2: 各参加者に4組ずつ配る（条件が重複しないように）
assignments = []
index = 0
max_tries = 1000

for attempt in range(max_tries):
    index = 0
    temp = []
    valid = True
    remaining = all_pairs.copy()
    for i in range(24):
        found = False
        for j in range(len(remaining) - 3):
            # 4つ連続でconditionがすべて異なるなら選ぶ
            candidate = remaining[j:j+4]
            conds = [c for (c, t) in candidate]
            if len(set(conds)) == 4:
                temp.append((participants[i], candidate))
                # 選んだ4つを取り除く
                for item in candidate:
                    remaining.remove(item)
                found = True
                break
        if not found:
            valid = False
            break
    if valid:
        assignments = temp
        break
else:
    raise Exception("条件の重複なし割り当てが見つかりませんでした")

# Step 3: 出力用データ作成
rows = []
for pid, pairs in assignments:
    random.shuffle(pairs)  # 各被験者内の順序もシャッフル
    row = {
        'Participant': pid,
    }
    for i, (cond, task) in enumerate(pairs, 1):
        row[f'Condition {i}'] = cond
        row[f'Task Set {i}'] = task
        row[f'Condition {i}_ConditionNum'] = conditions.index(cond) + 1
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("perfectly_balanced_assignment.csv", index=False, encoding="utf-8-sig")


Exception: 条件の重複なし割り当てが見つかりませんでした

In [2]:
!pip install ace_tools


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import json
import os

# 対象フォルダとファイル接頭文字
folder_path = "../InteractiveSmartHome/Assets/EXPERIMENT/ArrangeData"
letters = ['A', 'B', 'C', 'D']

for letter in letters:
    file_name = f"PreTaskArrangement{letter}.json"
    file_path = os.path.join(folder_path, file_name)

    # ファイル読み込み
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # task_id の追加
    for idx, item in enumerate(data):
        item["task_id"] = f"T{letter}{idx}"

    # 保存先ファイル名
    output_file = f"PreTaskArrangement{letter}.json"
    output_path = os.path.join(folder_path, output_file)

    # 保存
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    print(f"✅ {output_file} を保存しました")


✅ PreTaskArrangementA.json を保存しました
✅ PreTaskArrangementB.json を保存しました
✅ PreTaskArrangementC.json を保存しました
✅ PreTaskArrangementD.json を保存しました


In [9]:
import csv

# CSVに書き込むデータ（1行目はヘッダー）
header = ['ConditionID', 'ConditionName']
rows = [
    [1, 'Spatial Reference (Wizard of Oz)'],
    [2, 'Label (Wizard of Oz)'],
    [3, 'Pointing (Wizard of Oz)'],
    [4, 'Spatial Reference (System)']
]

# ファイル出力
with open('conditions.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(header)
    writer.writerows(rows)

print("✅ conditions.csv が作成されました。")


✅ conditions.csv が作成されました。


In [24]:
import pandas as pd
import random
import itertools
from collections import defaultdict

# --- 設定 ---
SEED = 42
random.seed(SEED)
rng = random.Random(SEED)  # タスクセット用に独立したRandomインスタンスを使う

participants = [f'P{i+1}' for i in range(24)]
conditions = ['spatial', 'pointing+spatial', 'pointing', 'label']
task_sets = ['A', 'B', 'C', 'D']
max_pair_count = 6  # 各 (condition, taskset) の最大出現数

# --- Step 1: ラテン方格を生成 ---
latin_square = [conditions[i:] + conditions[:i] for i in range(4)]
condition_orders = [latin_square[i % 4] for i in range(24)]  # 24人にラテン順序を割当

# --- Step 2: Condition × TaskSet 割当（完全バランス） ---
pair_counts = defaultdict(int)
final_assignments = []

# タスクセットのすべての順列を固定順で使う
candidate_tasksets_all = sorted(list(itertools.permutations(task_sets)))

for cond_order in condition_orders:
    candidate_tasksets = candidate_tasksets_all[:]
    rng.shuffle(candidate_tasksets)  # 固定シードでシャッフル

    for taskset_order in candidate_tasksets:
        temp_counts = pair_counts.copy()
        valid = True
        for c, t in zip(cond_order, taskset_order):
            if temp_counts[(c, t)] + 1 > max_pair_count:
                valid = False
                break
            temp_counts[(c, t)] += 1
        if valid:
            final_assignments.append((cond_order, taskset_order))
            pair_counts = temp_counts
            break
    else:
        raise Exception("完全割当できませんでした。シードを変更してください。")

# --- Step 3: DataFrame に整形 ---
rows = []
for i, (conds, tasks) in enumerate(final_assignments):
    row = {'Participant': f'P{i+1}'}
    for j, (c, t) in enumerate(zip(conds, tasks), start=1):
        row[f'Condition {j}'] = c
        row[f'Task Set {j}'] = t
        row[f'Condition {j}_ConditionNum'] = conditions.index(c) + 1
    rows.append(row)

df = pd.DataFrame(rows)

# --- Step 4: 保存（必要なら） ---
df.to_csv("latin_square_condition_task_balanced.csv", index=False, encoding="utf-8-sig")

# --- Step 5: 出現回数の確認 ---
counts = pd.DataFrame([
    (c, t) for conds, tasks in final_assignments for c, t in zip(conds, tasks)
], columns=['Condition', 'TaskSet']).value_counts().unstack().fillna(0).astype(int)

print(counts)


TaskSet           A  B  C  D
Condition                   
label             6  6  6  6
pointing          6  6  6  6
pointing+spatial  6  6  6  6
spatial           6  6  6  6
